# 10 - DCA EUR / Duration-Style Forecast

In this notebook, we use the existing DCA model fits to forecast oil production beyond the validation window.

The earlier benchmark notebooks used month 24 as the forecast origin and months 25-33 as the validation window.

This notebook asks a different practical DCA question: if we extend the fitted decline curves beyond the observed validation horizon, what EUR-style oil outcomes do they imply?

This is an oil-only release 1 workflow. Gas forecasting remains excluded.

Important reserves framing: the outputs in this notebook are not SPE PRMS reserves estimates. No economic limit, abandonment rate, commercial cutoff, or reserves-category assessment has been applied. These are technical modeled recoverable oil estimates under the stated DCA assumptions and forecast horizon.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "martin_selected_30_monthly_production_normalized.csv"
DCA_OUTPUT_DIR = PROJECT_ROOT / "reports" / "dca_outputs"

EXPONENTIAL_FILE = DCA_OUTPUT_DIR / "per_well_exponential_dca_results.csv"
HYPERBOLIC_FILE = DCA_OUTPUT_DIR / "per_well_hyperbolic_comparison_ready.csv"
HARMONIC_FILE = DCA_OUTPUT_DIR / "per_well_harmonic_comparison_ready.csv"

DATA_FILE.exists(), EXPONENTIAL_FILE.exists(), HYPERBOLIC_FILE.exists(), HARMONIC_FILE.exists()

## Forecast setup

We will keep the validation setup visible, but this notebook is not another benchmark scorecard.

- Forecast origin: month 24
- Validation window used earlier: months 25-33
- EUR-style forecast extension: months 34-120
- Primary horizon: month 120
- Optional duration-style threshold: first forecast month below 10 bbl/month

The 10 bbl/month threshold is a simple low-rate marker, not a full economic limit. A true economic limit would require price, cost, royalty, tax, and operating assumptions that are outside this release 1 workflow.

In [ ]:
FORECAST_ORIGIN_MONTH = 24
VALIDATION_START_MONTH = 25
VALIDATION_END_MONTH = 33

EUR_FORECAST_START_MONTH = 34
EUR_FORECAST_END_MONTH = 120

LOW_RATE_THRESHOLD_BBL_PER_MONTH = 10

In [ ]:
production = pd.read_csv(DATA_FILE)

production.shape

In [ ]:
production.head()

For this release, we only need the well identifiers, production month, and oil volume.

Gas columns are intentionally left out of the EUR-style forecast workflow.

In [ ]:
oil_history = production[
    [
        "api8",
        "lease_name",
        "well_no",
        "month_on_production",
        "oil_bbl",
    ]
].copy()

oil_history.head()

In [ ]:
oil_history["month_on_production"].agg(["min", "max"])

Now we load the existing DCA fit outputs.

These files already contain the fitted decline parameters from the earlier DCA notebooks. Notebook 10 will reuse those fits rather than refitting the models.

In [ ]:
exponential_fits = pd.read_csv(EXPONENTIAL_FILE)
hyperbolic_fits = pd.read_csv(HYPERBOLIC_FILE)
harmonic_fits = pd.read_csv(HARMONIC_FILE)

exponential_fits.shape, hyperbolic_fits.shape, harmonic_fits.shape

In [ ]:
exponential_fits.head()

In [ ]:
hyperbolic_fits.head()

In [ ]:
harmonic_fits.head()

The three DCA output files use slightly different column names.

Before forecasting, we will standardize each table to the same core fields:

- well identifiers
- model name
- fitted initial rate
- fitted decline rate
- fitted b-factor

In [ ]:
exponential_fits.columns.tolist()

In [ ]:
hyperbolic_fits.columns.tolist()

In [ ]:
harmonic_fits.columns.tolist()

For exponential decline, the b-factor is not used. We will store it as 0 so all model rows share the same structure.

For harmonic decline, the b-factor is 1.

In [ ]:
exponential_standard = exponential_fits[
    [
        "api8",
        "lease_name",
        "well_no",
        "qi_bbl_per_month",
        "di_per_month",
    ]
].copy()

exponential_standard["model"] = "exponential"
exponential_standard["b_factor"] = 0.0

exponential_standard.head()

In [ ]:
hyperbolic_standard = hyperbolic_fits[
    [
        "api8",
        "lease_name",
        "well_no",
        "hyperbolic_qi_bbl_per_month",
        "hyperbolic_di_per_month",
        "hyperbolic_b_factor",
    ]
].copy()

hyperbolic_standard = hyperbolic_standard.rename(
    columns={
        "hyperbolic_qi_bbl_per_month": "qi_bbl_per_month",
        "hyperbolic_di_per_month": "di_per_month",
        "hyperbolic_b_factor": "b_factor",
    }
)
hyperbolic_standard["model"] = "hyperbolic"

hyperbolic_standard.head()

In [ ]:
harmonic_standard = harmonic_fits[
    [
        "api8",
        "lease_name",
        "well_no",
        "harmonic_qi_bbl_per_month",
        "harmonic_di_per_month",
        "harmonic_b_factor",
    ]
].copy()

harmonic_standard = harmonic_standard.rename(
    columns={
        "harmonic_qi_bbl_per_month": "qi_bbl_per_month",
        "harmonic_di_per_month": "di_per_month",
        "harmonic_b_factor": "b_factor",
    }
)
harmonic_standard["model"] = "harmonic"

harmonic_standard.head()

In [ ]:
dca_fits = pd.concat(
    [
        exponential_standard,
        hyperbolic_standard,
        harmonic_standard,
    ],
    ignore_index=True,
)

dca_fits = dca_fits[
    [
        "api8",
        "lease_name",
        "well_no",
        "model",
        "qi_bbl_per_month",
        "di_per_month",
        "b_factor",
    ]
]

dca_fits.head()

In [ ]:
dca_fits["model"].value_counts()

## Fit quality check for forward EUR use

For a forward EUR-style DCA extension, the fitted decline rate must be positive.

If `di_per_month` is zero or negative, the forecast curve is flat or increasing. That can happen in a short noisy fit window, but it is not a valid decline assumption for month-120 EUR-style forecasting.

We will keep those rows visible, then exclude them from the forward EUR calculation.

In [ ]:
invalid_dca_fits = dca_fits[dca_fits["di_per_month"] <= 0].copy()

invalid_dca_fits

In [ ]:
dca_fits = dca_fits[dca_fits["di_per_month"] > 0].copy()

dca_fits["model"].value_counts()

## DCA forecast equations

Now we define the monthly oil-rate equations used for the forward forecast.

For exponential decline:

`q(t) = qi * exp(-di * t)`

For hyperbolic decline:

`q(t) = qi / (1 + b * di * t) ** (1 / b)`

Harmonic decline is the special case where `b = 1`.

Here, `t` is the well's month on production.

This matches the earlier DCA notebooks, where the decline curves were fit directly against `month_on_production`.

In [ ]:
def dca_monthly_oil_rate(qi_bbl_per_month, di_per_month, b_factor, month_on_production):
    if b_factor == 0:
        return qi_bbl_per_month * np.exp(-di_per_month * month_on_production)

    return qi_bbl_per_month / (1 + b_factor * di_per_month * month_on_production) ** (1 / b_factor)

Before forecasting the full EUR-style window, we test the function on one row.

This is a quick sanity check that the formula returns a positive monthly oil volume.

In [ ]:
example_fit = dca_fits.iloc[0]

example_fit

In [ ]:
dca_monthly_oil_rate(
    qi_bbl_per_month=example_fit["qi_bbl_per_month"],
    di_per_month=example_fit["di_per_month"],
    b_factor=example_fit["b_factor"],
    month_on_production=1,
)

The DCA fit files were created from the earlier workflow, where the fit window ended at month 24 and the validation forecast began at month 25.

Because the fitted equations use `month_on_production` directly, we forecast month 34 by passing `month_on_production = 34`, and month 120 by passing `month_on_production = 120`.

In [ ]:
example_target_month = EUR_FORECAST_START_MONTH

example_target_month

## Build the EUR-style forecast table

Now we create one row per well, model, and forecast month.

This table starts after the validation window. It covers months 34-120.

In [ ]:
forecast_months = np.arange(EUR_FORECAST_START_MONTH, EUR_FORECAST_END_MONTH + 1)

forecast_months[:5], forecast_months[-5:], len(forecast_months)

In [ ]:
forecast_rows = []

for _, fit in dca_fits.iterrows():
    for month in forecast_months:
        forecast_oil_bbl = dca_monthly_oil_rate(
            qi_bbl_per_month=fit["qi_bbl_per_month"],
            di_per_month=fit["di_per_month"],
            b_factor=fit["b_factor"],
            month_on_production=month,
        )

        forecast_rows.append(
            {
                "api8": fit["api8"],
                "lease_name": fit["lease_name"],
                "well_no": fit["well_no"],
                "model": fit["model"],
                "month_on_production": month,
                "forecast_oil_bbl": forecast_oil_bbl,
            }
        )

eur_forecast_monthly = pd.DataFrame(forecast_rows)

eur_forecast_monthly.head()

In [ ]:
eur_forecast_monthly.shape

In [ ]:
eur_forecast_monthly.groupby("model")["month_on_production"].agg(["min", "max", "count"])

## Historical cumulative oil

For this first EUR-style pass, historical cumulative oil is measured through month 33.

That keeps the handoff clean:

- months 1-24 were available before the fixed-origin forecast
- months 25-33 were the validation window
- months 34-120 are the DCA extension beyond validation

In [ ]:
historical_oil_through_validation = oil_history[
    oil_history["month_on_production"].between(1, VALIDATION_END_MONTH)
].copy()

historical_oil_through_validation.head()

In [ ]:
historical_cumulative_oil = (
    historical_oil_through_validation
    .groupby(["api8", "lease_name", "well_no"], as_index=False)
    .agg(historical_cumulative_oil_bbl=("oil_bbl", "sum"))
)

historical_cumulative_oil.head()

In [ ]:
historical_cumulative_oil["historical_cumulative_oil_bbl"].describe()

## Forecast cumulative oil

Next we sum the DCA forecast months 34-120 for each well and model.

In [ ]:
forecast_cumulative_oil = (
    eur_forecast_monthly
    .groupby(["api8", "lease_name", "well_no", "model"], as_index=False)
    .agg(forecast_cumulative_oil_bbl=("forecast_oil_bbl", "sum"))
)

forecast_cumulative_oil.head()

In [ ]:
forecast_cumulative_oil.groupby("model")["forecast_cumulative_oil_bbl"].describe()

## EUR-style oil total

Now we combine the observed oil through month 33 with the DCA forecast oil from months 34-120.

This is an EUR-style oil estimate through a fixed 120-month horizon.

It should be treated as technical modeled recoverable oil only. It is not an SPE PRMS reserves estimate because no economic limit, abandonment cutoff, commerciality screen, ownership adjustment, or reserves-category assessment has been applied.

In [ ]:
eur_summary = forecast_cumulative_oil.merge(
    historical_cumulative_oil,
    on=["api8", "lease_name", "well_no"],
    how="left",
)

eur_summary["eur_style_oil_bbl"] = (
    eur_summary["historical_cumulative_oil_bbl"]
    + eur_summary["forecast_cumulative_oil_bbl"]
)

eur_summary.head()

In [ ]:
eur_summary.groupby("model")[[
    "historical_cumulative_oil_bbl",
    "forecast_cumulative_oil_bbl",
    "eur_style_oil_bbl",
]].sum().round(0)

## Duration-style low-rate threshold

A fixed month-120 horizon gives us one EUR-style oil total.

We can also ask a simple duration-style question: when does each forecast curve first fall below 10 bbl/month?

This is not a full economic limit. It is only a transparent low-rate threshold.

In [ ]:
below_threshold = eur_forecast_monthly[
    eur_forecast_monthly["forecast_oil_bbl"] < LOW_RATE_THRESHOLD_BBL_PER_MONTH
].copy()

below_threshold.head()

In [ ]:
first_low_rate_month = (
    below_threshold
    .groupby(["api8", "lease_name", "well_no", "model"], as_index=False)
    .agg(first_month_below_threshold=("month_on_production", "min"))
)

first_low_rate_month.head()

In [ ]:
eur_summary = eur_summary.merge(
    first_low_rate_month,
    on=["api8", "lease_name", "well_no", "model"],
    how="left",
)

eur_summary["reaches_low_rate_threshold_by_month_120"] = eur_summary[
    "first_month_below_threshold"
].notna()

eur_summary.head()

In [ ]:
eur_summary.groupby("model").agg(
    wells=("api8", "count"),
    wells_reaching_threshold=("reaches_low_rate_threshold_by_month_120", "sum"),
    median_first_month_below_threshold=("first_month_below_threshold", "median"),
)

If `first_month_below_threshold` is blank, that model's forecast did not fall below the threshold by month 120.

## Review EUR-style results

Now we sort the per-well results so the largest EUR-style oil totals are easy to inspect.

In [ ]:
eur_summary_sorted = eur_summary.sort_values(
    ["model", "eur_style_oil_bbl"],
    ascending=[True, False],
).reset_index(drop=True)

eur_summary_sorted.head(10)

In [ ]:
eur_summary_sorted[
    [
        "api8",
        "lease_name",
        "well_no",
        "model",
        "historical_cumulative_oil_bbl",
        "forecast_cumulative_oil_bbl",
        "eur_style_oil_bbl",
        "first_month_below_threshold",
    ]
].head(15).round(0)

The same historical cumulative oil is repeated across models for each well.

The difference between models comes from the forecast extension from months 34-120.

In [ ]:
model_level_eur_summary = (
    eur_summary
    .groupby("model", as_index=False)
    .agg(
        wells=("api8", "count"),
        historical_cumulative_oil_bbl=("historical_cumulative_oil_bbl", "sum"),
        forecast_cumulative_oil_bbl=("forecast_cumulative_oil_bbl", "sum"),
        eur_style_oil_bbl=("eur_style_oil_bbl", "sum"),
        median_well_eur_style_oil_bbl=("eur_style_oil_bbl", "median"),
        wells_reaching_low_rate_threshold=("reaches_low_rate_threshold_by_month_120", "sum"),
        median_first_month_below_threshold=("first_month_below_threshold", "median"),
    )
)

model_level_eur_summary.round(0)

In [ ]:
model_level_eur_summary.sort_values("eur_style_oil_bbl", ascending=False).round(0)

## Simple 30-well technical recovery table

For reporting, it is useful to have one compact table with one row per well or producing conduit.

In this selected dataset, all 30 wells are labeled under `SPRABERRY (TREND AREA)`. If future data splits a well across multiple reservoir targets or conduits, this table should be rebuilt at that finer well-target level.

The table below keeps the model-specific technical recoverable oil estimates side by side. Blank model values mean that model was excluded for that well by the fit-quality screen.

In [ ]:
well_target_lookup = (
    production
    .groupby(["api8", "lease_name", "well_no"], as_index=False)
    .agg(reservoir_or_conduit_target=("field_name", "first"))
)

well_target_lookup["well_name"] = (
    well_target_lookup["lease_name"] + " " + well_target_lookup["well_no"].astype(str)
)

well_target_lookup.head()

In [ ]:
well_technical_recovery_table = historical_cumulative_oil.merge(
    well_target_lookup,
    on=["api8", "lease_name", "well_no"],
    how="left",
)

model_eur_wide = eur_summary.pivot_table(
    index=["api8", "lease_name", "well_no"],
    columns="model",
    values="eur_style_oil_bbl",
    aggfunc="first",
).reset_index()

model_eur_wide = model_eur_wide.rename(
    columns={
        "exponential": "exponential_technical_recoverable_oil_bbl",
        "hyperbolic": "hyperbolic_technical_recoverable_oil_bbl",
        "harmonic": "harmonic_technical_recoverable_oil_bbl",
    }
)

well_technical_recovery_table = well_technical_recovery_table.merge(
    model_eur_wide,
    on=["api8", "lease_name", "well_no"],
    how="left",
)

well_technical_recovery_table.head()

In [ ]:
well_technical_recovery_table = well_technical_recovery_table[
    [
        "api8",
        "well_name",
        "lease_name",
        "well_no",
        "reservoir_or_conduit_target",
        "historical_cumulative_oil_bbl",
        "exponential_technical_recoverable_oil_bbl",
        "hyperbolic_technical_recoverable_oil_bbl",
        "harmonic_technical_recoverable_oil_bbl",
    ]
].sort_values("hyperbolic_technical_recoverable_oil_bbl", ascending=False)

well_technical_recovery_table.round(0)

In [ ]:
well_technical_recovery_table.shape

## Visual review

These quick plots help make the fixed-horizon EUR-style results easier to inspect.

In [ ]:
plot_model_summary = model_level_eur_summary.sort_values("eur_style_oil_bbl", ascending=False)

if plt is None:
    plot_model_summary
else:
    plt.figure(figsize=(8, 5))
    plt.bar(
        plot_model_summary["model"],
        plot_model_summary["eur_style_oil_bbl"] / 1_000_000,
    )
    plt.title("DCA EUR-Style Oil Through Month 120")
    plt.xlabel("DCA model")
    plt.ylabel("Oil, million bbl")
    plt.grid(axis="y", alpha=0.30)
    plt.tight_layout()

Next we inspect one example well.

The example is selected from wells that have all three model forecasts available after the fit-quality filter.

In [ ]:
valid_model_count_by_well = (
    eur_summary
    .groupby(["api8", "lease_name", "well_no"], as_index=False)
    .agg(model_count=("model", "nunique"))
)

example_well = valid_model_count_by_well[
    valid_model_count_by_well["model_count"] == 3
].iloc[0]

example_well

In [ ]:
example_history = oil_history[
    (oil_history["api8"] == example_well["api8"])
    & (oil_history["month_on_production"] <= VALIDATION_END_MONTH)
].copy()

example_forecast = eur_forecast_monthly[
    eur_forecast_monthly["api8"] == example_well["api8"]
].copy()

if plt is None:
    example_forecast.head(15)
else:
    plt.figure(figsize=(10, 6))
    plt.plot(
        example_history["month_on_production"],
        example_history["oil_bbl"],
        marker="o",
        label="Observed oil through month 33",
    )

    for model_name, model_forecast in example_forecast.groupby("model"):
        plt.plot(
            model_forecast["month_on_production"],
            model_forecast["forecast_oil_bbl"],
            linestyle="--",
            label=f"{model_name} forecast",
        )

    plt.axvline(VALIDATION_END_MONTH, color="black", linestyle=":", alpha=0.70)
    plt.title(f"Example DCA Extension: {example_well['lease_name']} {example_well['well_no']}")
    plt.xlabel("Month on production")
    plt.ylabel("Oil, bbl/month")
    plt.legend()
    plt.grid(alpha=0.30)
    plt.tight_layout()

## Export notebook 10 outputs

Finally, we save the monthly forecast table, the per-well EUR-style summary, the simple 30-well technical recovery table, the model-level summary, and the excluded-fit QA table.

These are DCA forecast-extension outputs, not validation-score outputs.

In [ ]:
EUR_MONTHLY_FORECAST_FILE = DCA_OUTPUT_DIR / "dca_eur_duration_monthly_forecast.csv"
EUR_PER_WELL_SUMMARY_FILE = DCA_OUTPUT_DIR / "dca_eur_duration_per_well_summary.csv"
EUR_30_WELL_TABLE_FILE = DCA_OUTPUT_DIR / "dca_30_well_technical_recoverable_oil_table.csv"
EUR_MODEL_SUMMARY_FILE = DCA_OUTPUT_DIR / "dca_eur_duration_model_summary.csv"
EUR_EXCLUDED_FITS_FILE = DCA_OUTPUT_DIR / "dca_eur_duration_excluded_fits.csv"

(
    EUR_MONTHLY_FORECAST_FILE,
    EUR_PER_WELL_SUMMARY_FILE,
    EUR_30_WELL_TABLE_FILE,
    EUR_MODEL_SUMMARY_FILE,
    EUR_EXCLUDED_FITS_FILE,
)

In [ ]:
eur_forecast_monthly.to_csv(EUR_MONTHLY_FORECAST_FILE, index=False)
eur_summary_sorted.to_csv(EUR_PER_WELL_SUMMARY_FILE, index=False)
well_technical_recovery_table.to_csv(EUR_30_WELL_TABLE_FILE, index=False)
model_level_eur_summary.to_csv(EUR_MODEL_SUMMARY_FILE, index=False)
invalid_dca_fits.to_csv(EUR_EXCLUDED_FITS_FILE, index=False)

In [ ]:
(
    EUR_MONTHLY_FORECAST_FILE.exists(),
    EUR_PER_WELL_SUMMARY_FILE.exists(),
    EUR_30_WELL_TABLE_FILE.exists(),
    EUR_MODEL_SUMMARY_FILE.exists(),
    EUR_EXCLUDED_FITS_FILE.exists(),
)

## Notebook 10 summary

This notebook extended the existing DCA model fits beyond the validation window and produced EUR-style oil totals through month 120.

Key interpretation points:

- Months 25-33 were used earlier for validation scoring.
- Months 34-120 are a forward DCA extension beyond that validation window.
- The workflow is oil-only; gas remains excluded from release 1.
- Non-declining fitted curves were excluded from the forward EUR extension.
- The low-rate threshold is a transparent duration marker, not a full economic limit.
- No SPE PRMS reserves classification has been applied.
- No economic limit, abandonment rate, or commercial cutoff has been implemented.
- The EUR-style totals are technical modeled recoverable oil estimates under the stated assumptions, not reserves estimates.